# Phase 01 — Three-Class Synthetic Image Classification

**Research task:** Classify images as **Real**, **GAN-generated**, or **Diffusion-generated** using PiD reconstruction residuals and handcrafted signal features with an attention-enhanced CNN.

**Recorded result:** 95.73% validation accuracy on 1,500 balanced images. The uploaded notebook output **loaded an existing checkpoint** and evaluated it; the captured session did not perform fresh training epochs.

**Dataset reference:** [GenImage](https://github.com/GenImage-Dataset/GenImage) ([website](https://genimage-dataset.github.io/)); the exact local three-class selection is not recorded in this notebook.

**Before running:** Change the Windows `WORK_DIR` path, prepare the selected GenImage images locally, and review the label-inference rules. All saved code outputs have been moved to `../results/phase-01/`.

Full project details: [Phase 01 README](README.md).

## 1. Check existing checkpoint (optional)
This cell reports whether a previous checkpoint is present. It is not a training cell.

In [ ]:
# Resume diagnostic: run this before the main training cell
import os
import torch

WORK_DIR = r"C:\Dataset\hybrid_pid_full_data"
MODEL_PATH = os.path.join(WORK_DIR, "best_model.pth")
CKPT_PATH = os.path.join(WORK_DIR, "checkpoint_latest.pth")
EPOCHS = 10

print("=" * 70)
print("RESUME DIAGNOSTIC")
print("=" * 70)
print(f"WORK_DIR: {WORK_DIR}")
print(f"Checkpoint exists: {os.path.exists(CKPT_PATH)}")
print(f"Best model exists: {os.path.exists(MODEL_PATH)}")

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location="cpu")
    ckpt_epoch = int(ckpt.get("epoch", -1))
    ckpt_best = float(ckpt.get("best_acc", 0.0))
    start_epoch = ckpt_epoch + 1

    print(f"Checkpoint epoch: {ckpt_epoch}")
    print(f"Will resume from epoch: {start_epoch}")
    print(f"Checkpoint best_acc: {ckpt_best:.4f}")

    if start_epoch >= EPOCHS:
        print("⚠️ No training will run because start_epoch >= EPOCHS.")
        print("   Increase EPOCHS (e.g., EPOCHS = start_epoch + 5) to continue training.")
    else:
        print(f"✅ Resume is valid. It will train epochs {start_epoch} .. {EPOCHS - 1}.")
else:
    print("⚠️ No checkpoint found. Your current code will train from scratch.")
    if os.path.exists(MODEL_PATH):
        print("ℹ️ best_model.pth exists, but your resume logic does NOT use it for continuation.")

print("=" * 70)


## 2. Imports and configuration
Edit `WORK_DIR`, `BATCH_SIZE`, and other settings before running.

In [ ]:
import os, io, json, shutil, random, traceback, sys
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image
from tqdm import tqdm
from datetime import datetime
import time
import matplotlib.pyplot as plt

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 10
SEED = 42
NUM_CLASSES = 3
CLASS_NAMES = ['Real', 'GAN', 'Diffusion']
WORK_DIR = r"C:\Dataset\hybrid_pid_full_data"
MODEL_PATH = os.path.join(WORK_DIR, "best_model.pth")
CKPT_PATH = os.path.join(WORK_DIR, "checkpoint_latest.pth")
BEST_META_PATH = os.path.join(WORK_DIR, "best_model_meta.json")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.makedirs(WORK_DIR, exist_ok=True)


## 3. Logging utilities

In [ ]:
# ==========================================
# LOGGING UTILITIES
# ==========================================
def log_msg(msg, flush=True):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {msg}")
    if flush:
        sys.stdout.flush()

start_time = time.time()
log_msg("=" * 70)
log_msg("DEEPFAKE DETECTION PIPELINE STARTING")
log_msg("=" * 70)
log_msg(f"Setup: Device={DEVICE}")
log_msg(f"Config: IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}, NUM_CLASSES={NUM_CLASSES}")
log_msg(f"Checkpoint: {CKPT_PATH}")
log_msg("  - On first run: Creates checkpoint every epoch + final checkpoint")
log_msg("  - On subsequent runs: AUTO-RESUMES from last checkpoint")
log_msg("=" * 70)


## 4. PiD and signal feature extraction

In [ ]:
# ==========================================
# FEATURE EXTRACTORS (PiD + Signal)
# ==========================================
def extract_pid_features(x):
    x255 = x * 255.0
    MYUV = torch.tensor([[0.299, 0.587, 0.114], [-0.168736, -0.331264, 0.5], [0.5, -0.418688, -0.081312]], dtype=torch.float32, device=x.device)
    MINv = torch.tensor([[1.0, 0.0, 1.402], [1.0, -0.344136, -0.714136], [1.0, 1.772, 0.0]], dtype=torch.float32, device=x.device)
    p = x255.permute(0, 2, 3, 1)
    recon = torch.round(torch.matmul(torch.round(torch.matmul(p, MYUV.T)), MINv.T))
    return ((p - recon).permute(0, 3, 1, 2)) / 255.0


def extract_signal_features(x):
    n = (x - 0.5) / 0.5
    k = torch.tensor([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]], dtype=torch.float32, device=x.device).view(1, 1, 3, 3) / 8.0
    noise = torch.cat([F.conv2d(n[:, i:i+1], k, padding=1) for i in range(3)], dim=1)
    gray = 0.299 * n[:, 0] + 0.587 * n[:, 1] + 0.114 * n[:, 2]
    gs = F.interpolate(gray.unsqueeze(1), (112, 112), mode='bilinear', align_corners=False).squeeze(1)
    mag = torch.log(torch.abs(torch.fft.fftshift(torch.fft.fftn(gs, dim=(-2, -1)), dim=(-2, -1))) + 1e-8)
    mn, mx = mag.amin(dim=(-2, -1), keepdim=True), mag.amax(dim=(-2, -1), keepdim=True)
    fft = F.interpolate(((mag - mn) / (mx - mn + 1e-8)).unsqueeze(1), (IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False).repeat(1, 3, 1, 1)
    mean = n.mean(dim=[2, 3], keepdim=True).repeat(1, 1, IMG_SIZE, IMG_SIZE)
    std = n.std(dim=[2, 3], keepdim=True).repeat(1, 1, IMG_SIZE, IMG_SIZE)
    return torch.cat([noise, fft, mean, std], dim=1)


def get_feats(x):
    return torch.cat([extract_pid_features(x), extract_signal_features(x)], dim=1)


## 5. Attention-enhanced CNN architecture

In [ ]:
# ==========================================
# MODEL ARCHITECTURE
# ==========================================
class SEBlock(nn.Module):
    def __init__(self, c, r=4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(c, c // r, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(c // r, c, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)


class AttentionHybridClassifier(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(15, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2), SEBlock(64),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2), SEBlock(128),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.head(self.net(x).view(x.size(0), -1))


## 6. Dataset preparation and three-class labels

In [ ]:
# ==========================================
# DATASET LABELING (3 classes)
# ==========================================
GAN_MARKERS = ('_biggan', 'biggan', 'progan', 'stylegan', 'sngan', 'wgan', 'dcgan', 'cyclegan', 'stargan', 'cgan', 'gan')
DIFF_MARKERS = ('diffusion', 'ddpm', 'ldm', 'glide', 'dalle', 'stable', 'sd', 'imagen', 'vqdm', 'vq-dm')


class RawImageDataset(Dataset):
    def __init__(self, paths, labels, tf):
        self.paths, self.labels, self.tf = paths, labels, tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            return self.tf(Image.open(self.paths[i]).convert('RGB')), self.labels[i]
        except Exception:
            return torch.zeros(3, IMG_SIZE, IMG_SIZE), self.labels[i]


def infer_three_class_label(path):
    p_norm = path.replace('\\', '/').lower()
    parts = [p for p in p_norm.split('/') if p]

    if 'nature' in parts or 'real' in p_norm or '/0_real/' in p_norm:
        return 0
    if any(marker in p_norm for marker in GAN_MARKERS):
        return 1
    if any(marker in p_norm for marker in DIFF_MARKERS):
        return 2
    if 'fake' in parts or 'synthetic' in parts:
        return 2
    return None


def load_dataset_from_directory(root):
    log_msg(f"Scanning dataset from {root}...")
    categories = {0: [], 1: [], 2: []}

    for r, d, fs in os.walk(root):
        for f in fs:
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                full_path = os.path.join(r, f)
                label = infer_three_class_label(full_path)
                if label is not None:
                    categories[label].append(full_path)

    log_msg("Dataset counts discovered:")
    log_msg(f"  Real: {len(categories[0])}")
    log_msg(f"  GAN: {len(categories[1])}")
    log_msg(f"  Diffusion: {len(categories[2])}")

    min_count = min(len(categories[0]), len(categories[1]), len(categories[2]))
    if min_count == 0:
        raise RuntimeError("One or more classes have zero images after filtering. Check path rules in infer_three_class_label().")

    for label in categories:
        categories[label] = random.sample(categories[label], min_count)

    log_msg(f"Balanced to {min_count} per class (Total: {min_count * 3} images)")
    return categories[0], categories[1], categories[2]


## 7. Checkpoint utilities

In [ ]:
# ==========================================
# CHECKPOINT MANAGEMENT
# ==========================================
def save_checkpoint(epoch, best_acc):
    torch.save({
        'epoch': epoch,
        'model': model.state_dict(),
        'optimizer': opt.state_dict(),
        'scheduler': sched.state_dict(),
        'scaler': scaler.state_dict(),
        'best_acc': best_acc,
        'num_classes': NUM_CLASSES,
    }, CKPT_PATH)
    log_msg(f"  Checkpoint saved (epoch {epoch + 1})")


## 8. Train, validate, and report metrics

In [ ]:
# ==========================================
# TRAINING PIPELINE
# ==========================================
log_msg("\n" + "=" * 70)
log_msg("PHASE 1: Data Loading & Preparation")
log_msg("=" * 70)

real_paths, gan_paths, diff_paths = load_dataset_from_directory(os.path.join(WORK_DIR, ".."))
log_msg(f"Total: {len(real_paths)} real, {len(gan_paths)} GAN, {len(diff_paths)} diffusion")

log_msg("Splitting data: 80% train, 20% val...")
real_tr, real_val = train_test_split(real_paths, test_size=0.2, random_state=SEED)
gan_tr, gan_val = train_test_split(gan_paths, test_size=0.2, random_state=SEED)
diff_tr, diff_val = train_test_split(diff_paths, test_size=0.2, random_state=SEED)

tr_paths = real_tr + gan_tr + diff_tr
tr_labels = [0] * len(real_tr) + [1] * len(gan_tr) + [2] * len(diff_tr)
val_paths = real_val + gan_val + diff_val
val_labels = [0] * len(real_val) + [1] * len(gan_val) + [2] * len(diff_val)

log_msg(f"Train: {len(tr_paths)} images | Val: {len(val_paths)} images")

log_msg("Creating weighted sampler for balanced training...")
sampler = WeightedRandomSampler(torch.tensor([1.0 / np.bincount(tr_labels)[l] for l in tr_labels]), len(tr_labels))

log_msg("Building DataLoaders...")
tr_loader = DataLoader(RawImageDataset(tr_paths, tr_labels, T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor()])), batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(RawImageDataset(val_paths, val_labels, T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor()])), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

log_msg(f"Batches per epoch: train={len(tr_loader)}, val={len(val_loader)}")

log_msg("\n" + "=" * 70)
log_msg("PHASE 2: Model Initialization & Training Setup")
log_msg("=" * 70)
log_msg("Building model...")

model = AttentionHybridClassifier(num_classes=NUM_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
log_msg(f"  Model Parameters: {total_params:,}")

log_msg("Setting up loss function, optimizer, scheduler...")
crit = nn.CrossEntropyLoss(label_smoothing=0.1)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt,
    max_lr=1e-3,
    steps_per_epoch=len(tr_loader),
    epochs=EPOCHS,
    pct_start=0.1,
    div_factor=25,
    final_div_factor=1e4,
)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))
best_acc = 0.0
start_epoch = 0

log_msg("Checking for checkpoint...")
if os.path.exists(CKPT_PATH):
    log_msg(f"  Found checkpoint: {CKPT_PATH}")
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    if int(ckpt.get('num_classes', NUM_CLASSES)) == NUM_CLASSES:
        try:
            model.load_state_dict(ckpt['model'])
            opt.load_state_dict(ckpt['optimizer'])
            sched.load_state_dict(ckpt['scheduler'])
            scaler.load_state_dict(ckpt['scaler'])
            best_acc = float(ckpt.get('best_acc', 0.0))
            start_epoch = int(ckpt.get('epoch', -1)) + 1
            log_msg(f"RESUMING from epoch {start_epoch}/{EPOCHS}, best_acc={best_acc:.4f}")
        except Exception as exc:
            log_msg(f"Checkpoint load failed, starting from scratch: {exc}")
    else:
        log_msg("Checkpoint was trained with a different class count; starting from scratch.")
elif os.path.exists(MODEL_PATH):
    log_msg(f"  Found best model weights: {MODEL_PATH}")
    try:
        model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
        if os.path.exists(BEST_META_PATH):
            with open(BEST_META_PATH, 'r') as f:
                meta = json.load(f)
            if int(meta.get('num_classes', NUM_CLASSES)) == NUM_CLASSES:
                best_acc = float(meta.get('best_acc', 0.0))
                start_epoch = int(meta.get('epoch', -1)) + 1
                log_msg(f"Warm-start from best_model. Approx epoch={start_epoch}, best_acc={best_acc:.4f}")
            else:
                log_msg("Best-model metadata says a different class count; starting from scratch.")
        else:
            log_msg("Warm-start from best_model (no epoch metadata found, starting from epoch 1).")
    except Exception as exc:
        log_msg(f"Best model incompatible with 3-class head; starting from scratch: {exc}")
else:
    log_msg("Training from scratch")

if start_epoch >= EPOCHS:
    log_msg(f"start_epoch ({start_epoch}) >= EPOCHS ({EPOCHS}). Increase EPOCHS to continue training.")

log_msg("=" * 70)
log_msg("STARTING TRAINING LOOP")
log_msg("=" * 70)

for epoch in range(start_epoch, EPOCHS):
    log_msg(f"\nEPOCH {epoch + 1}/{EPOCHS}")
    epoch_start = time.time()

    log_msg("  Training...")
    model.train()
    train_loss = 0.0
    batch_count = 0

    for x_raw, y in tqdm(tr_loader, desc="      Train", leave=False):
        x_raw = x_raw.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with torch.no_grad():
            feats = get_feats(x_raw)

        opt.zero_grad()
        with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda')):
            pred = model(feats)
            loss = crit(pred, y)

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()

        train_loss += loss.item()
        batch_count += 1

    avg_train_loss = train_loss / max(batch_count, 1)

    log_msg("  Validating...")
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x_raw, y in tqdm(val_loader, desc="      Val", leave=False):
            x_raw = x_raw.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            feats = get_feats(x_raw)
            correct += (model(feats).argmax(1) == y).sum().item()
            total += y.size(0)

    acc = correct / max(total, 1)
    lr = opt.param_groups[0]['lr']
    epoch_time = time.time() - epoch_start

    log_msg(f"  Loss={avg_train_loss:.4f} | Val Acc={acc:.4f} | LR={lr:.2e} | Time={epoch_time:.1f}s")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), MODEL_PATH)
        with open(BEST_META_PATH, 'w') as f:
            json.dump({'epoch': epoch, 'best_acc': float(best_acc), 'num_classes': NUM_CLASSES}, f)
        log_msg(f"  NEW BEST: {best_acc:.4f}")

    save_checkpoint(epoch, best_acc)

save_checkpoint(EPOCHS - 1, best_acc)
log_msg(f"\nTraining complete. Best Val Acc: {best_acc:.4f}")

log_msg("\n" + "=" * 70)
log_msg("PHASE 3: Final Summary & Metrics")
log_msg("=" * 70)

model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

log_msg("Computing final metrics...")
y_true, y_pred = [], []
with torch.no_grad():
    for x_raw, y in val_loader:
        x_raw = x_raw.to(DEVICE, non_blocking=True)
        feats = get_feats(x_raw)
        y_pred.extend(model(feats).argmax(1).cpu().numpy())
        y_true.extend(y.numpy())

log_msg("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(y_true, y_pred)
log_msg("\nConfusion Matrix:")
log_msg(f"{'':10} {CLASS_NAMES[0]:<10} {CLASS_NAMES[1]:<10} {CLASS_NAMES[2]:<10}")
for i, name in enumerate(CLASS_NAMES):
    log_msg(f"{name:<10} {cm[i,0]:<10} {cm[i,1]:<10} {cm[i,2]:<10}")

class_totals = cm.sum(axis=1)
class_acc = np.divide(cm.diagonal(), class_totals, out=np.zeros_like(cm.diagonal(), dtype=float), where=class_totals != 0)

plt.figure(figsize=(10, 7))
bars = plt.bar(CLASS_NAMES, class_acc, color=['#4C72B0', '#DD8452', '#55A868'])
plt.ylim(0, 1.15)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Per-Class Accuracy', fontsize=14, pad=20)
plt.xticks(rotation=45, ha='right')
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height + 0.03, f'{height:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=11)
plt.tight_layout()
chart_path = os.path.join(WORK_DIR, 'per_class_accuracy.png')
plt.savefig(chart_path, dpi=300, bbox_inches='tight')
log_msg(f"Chart saved: {chart_path}")
plt.show()

total_time = (time.time() - start_time) / 60
log_msg(f"\nTotal execution time: {total_time:.1f} minutes")
log_msg(f"Best Validation Accuracy: {best_acc:.4f}")
log_msg(f"Model saved: {MODEL_PATH}")
log_msg(f"Checkpoint: {CKPT_PATH}")
log_msg("=" * 70)
log_msg("ALL DONE")
log_msg("=" * 70)


## 9. Parameter count

In [ ]:
# Parameter count for the current model
if 'model' not in globals():
    raise RuntimeError('Run the main training cell first so model exists in memory.')

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('=' * 70)
print('MODEL PARAMETER COUNT')
print('=' * 70)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Frozen parameters:    {total_params - trainable_params:,}')
print('=' * 70)
